In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import io
import re

In [2]:
# === CONFIGURAÇÕES ===
# Caminho para seu CSV (pode ser o summary consolidado)
CSV_PATH = "./summary_results.csv"

In [3]:
# === LEITURA E LIMPEZA ===
# Tenta ler diretamente tratando vírgula decimal; se falhar, faz fallback lendo o arquivo e substituindo vírgula por ponto.

try:
    df = pd.read_csv(CSV_PATH, encoding='utf-8', decimal=",")
except Exception:
    with open(CSV_PATH, "r", encoding="utf-8") as f:
        content = f.read().replace(",", ".")
    df = pd.read_csv(io.StringIO(content))
# === PRÉ-PROCESSAMENTO ===
# Garante nomes consistentes
df.columns = [c.strip().replace('"', '') for c in df.columns]
# Remove espaços, corrige tipos numéricos
for col in ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Extrai linguagem e teste do nome do executável (ex: tc1_HS.exe)
def parse_test_lang(filename):
    base = str(filename).replace(".exe", "").replace(".EXE", "")
    parts = base.split("_")
    if len(parts) >= 2:
        test = "_".join(parts[:-1])
        lang = parts[-1]
        return test, lang
    return base, "Unknown"

df[["Test", "Language"]] = df["File"].apply(lambda f: pd.Series(parse_test_lang(f)))

# Extrai nível de otimização (o1, o2, o3...) do nome do assembly
df["Optimization"] = df["AsmFile"].apply(
    lambda x: re.search(r"o(\d+)", str(x)).group(1) if re.search(r"o(\d+)", str(x)) else "0"
)


In [4]:
# === DEBUG ===
print("\n=== DADOS FORMATADOS ===")
print(df.head(5))
print("\nLinguagens:", df["Language"].unique())
print("Testes:", df["Test"].unique())
print("Otimizacoes:", df["Optimization"].unique())

# === VISÃO GERAL ===
print("\n=== RESUMO DOS DADOS ===")
print(df.head())
print("\nLinguagens:", df['Language'].unique())
print("Testes:", df['Test'].unique())
print("Assembly Files:", df['AsmFile'].unique())



=== DADOS FORMATADOS ===
        File  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmFile  \
0  mpf_c.exe       177.18          2.008          0.422        1.73  mpf_c.s   
1  mpf_c.exe       165.44          2.008          0.414        0.87  mpf_c.s   
2  mpf_c.exe       176.95          2.008          0.422        1.74  mpf_c.s   
3  mpf_c.exe       163.69          2.008          0.410        0.00  mpf_c.s   
4  mpf_c.exe       247.14          2.801          0.539        0.65  mpf_c.s   

   AsmLines Test Language Optimization  
0       292  mpf        c            0  
1       292  mpf        c            0  
2       292  mpf        c            0  
3       292  mpf        c            0  
4       292  mpf        c            0  

Linguagens: ['c' 'hs' 'py']
Testes: ['mpf']
Otimizacoes: ['0' '1' '2' '3']

=== RESUMO DOS DADOS ===
        File  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmFile  \
0  mpf_c.exe       177.18          2.008          0.422        1.

In [5]:
# === FUNÇÃO DE COMPARAÇÃO (adiciona boxplots, violin plots e gráficos de linha) ===
def compare_test2(test_name):
    """Gera gráficos comparando C e Haskell para um teste específico (usa média por Optimization).
       Além dos gráficos agregados, gera boxplots, violin plots e gráficos de linha com os dados individuais por nível de otimização."""
    subset = df[df["Test"] == test_name].copy()

    if subset.empty:
        print(f"⚠️ Nenhum dado encontrado para o teste '{test_name}'")
        return

    metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
    metric_labels = {
        "ExecTime_ms": "Tempo de Execução (ms)",
        "MaxWorking_MB": "Memória Física Máx (MB)",
        "MaxPrivate_MB": "Memória Privada Máx (MB)",
        "AvgCPU_pct": "Uso de CPU (%)",
        "AsmLines": "Linhas Assembly"
    }

    # Garantir que Optimization seja ordenável (int) para um eixo X correto
    subset["OptLevel"] = pd.to_numeric(subset["Optimization"], errors="coerce").fillna(-1).astype(int)
    subset.sort_values("OptLevel", inplace=True)
    subset["Optimization_str"] = subset["OptLevel"].astype(str)

    # Para cada métrica, plotar a média por (Optimization, Language) com barras (e erro padrão) e linhas
    for metric in metrics:
        agg = subset.groupby(["OptLevel", "Language"])[metric].agg(["mean", "std", "count"]).reset_index()
        if not agg.empty:
            agg["Optimization"] = agg["OptLevel"].astype(str)

            # Bar chart (média por optimization)
            fig = px.bar(
                agg,
                x="Optimization",
                y="mean",
                color="Language",
                barmode="group",
                error_y="std",
                labels={"mean": metric_labels[metric]},
                title=f"{test_name} — {metric_labels[metric]} (média por nível de otimização) — Barras",
            )
            fig.update_layout(
                template="plotly_white",
                font=dict(size=14),
                yaxis_title=metric_labels[metric],
                xaxis_title="Nível de Otimização"
            )
            fig.show()

            # Line chart (média por optimization) — adicionado
            fig_line = px.line(
                agg,
                x="Optimization",
                y="mean",
                color="Language",
                markers=True,
                labels={"mean": metric_labels[metric]},
                title=f"{test_name} — {metric_labels[metric]} (média por nível de otimização) — Linha",
            )
            fig_line.update_traces(mode="lines+markers")
            fig_line.update_layout(
                template="plotly_white",
                font=dict(size=14),
                yaxis_title=metric_labels[metric],
                xaxis_title="Nível de Otimização"
            )
            fig_line.show()

    # Boxplots e Violin plots usando dados individuais (por nível de otimização e linguagem)
    for metric in metrics:
        if metric not in subset.columns:
            continue
        # Boxplot
        fig_box = px.box(
            subset,
            x="Optimization_str",
            y=metric,
            color="Language",
            points="outliers",
            labels={metric: metric_labels[metric], "Optimization_str": "Nível de Otimização"},
            title=f"{test_name} — Boxplot de {metric_labels[metric]} por Nível de Otimização"
        )
        fig_box.update_layout(template="plotly_white", font=dict(size=13))
        fig_box.show()

        # Violin plot (com box interno e pontos)
        fig_violin = px.violin(
            subset,
            x="Optimization_str",
            y=metric,
            color="Language",
            box=True,
            points="all",
            labels={metric: metric_labels[metric], "Optimization_str": "Nível de Otimização"},
            title=f"{test_name} — Violin de {metric_labels[metric]} por Nível de Otimização"
        )
        fig_violin.update_layout(template="plotly_white", font=dict(size=13))
        fig_violin.show()

    # Comparativo médio por linguagem (usado para resumo, radar e speed ratio)
    mean_data = subset.groupby("Language")[metrics].mean().reset_index()
    print(f"=== MÉDIAS — {test_name} ===")
    print(mean_data)
    print()

    # Radar Chart (perfil de desempenho) - normalização segura (evita divisão por zero)
    normalized = mean_data.copy()
    for col in metrics:
        mn = mean_data[col].min()
        mx = mean_data[col].max()
        if pd.isna(mn) or pd.isna(mx) or mx == mn:
            normalized[col] = 0.5
        else:
            normalized[col] = (mean_data[col] - mn) / (mx - mn)

    fig_radar = go.Figure()
    for _, row in normalized.iterrows():
        fig_radar.add_trace(go.Scatterpolar(
            r=row[metrics].values,
            theta=[metric_labels[m] for m in metrics],
            fill="toself",
            name=row["Language"]
        ))
    fig_radar.update_layout(
        title=f"Perfil de Desempenho Normalizado — {test_name}",
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        template="plotly_white"
    )
    fig_radar.show()

    # Speed Ratio (quanto Haskell é mais lento/rápido que C) usando médias
    if {"C", "Haskell"}.issubset(set(mean_data["Language"])):
        c_data = mean_data.loc[mean_data["Language"] == "C"].iloc[0]
        hs_data = mean_data.loc[mean_data["Language"] == "Haskell"].iloc[0]

        if c_data["ExecTime_ms"] == 0:
            print(f"⚠️ ExecTime_ms média de C é zero para {test_name}, não é possível calcular ratio.")
        else:
            ratio = hs_data["ExecTime_ms"] / c_data["ExecTime_ms"]
            print(f"⚡ Speed Ratio ({test_name}): Haskell é {ratio:.2f}x o tempo de C.\n")


# === GERAR COMPARAÇÕES PARA TODOS OS TESTES ===
tests = df["Test"].unique()

for t in tests:
    compare_test2(t)

C:\Users\maria\AppData\Roaming\Python\Python313\site-packages\kaleido\__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




=== MÉDIAS — mpf ===
  Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  AsmLines
0        c   188.862333       2.132467       0.436817    0.753500    246.25
1       hs   183.846667       2.218333       3.276350    1.125500   1214.25
2       py   835.003333       6.352667       1.311733    2.663333      0.00



In [6]:
def compare_tests_aggregated(df):
    """
    Gera gráficos comparando a média das métricas entre diferentes Testes 
    (por exemplo, min1, tc1) para C e Haskell, ignorando o nível de otimização
    na agregação.
    """
    if df.empty:
        print("⚠️ O DataFrame está vazio.")
        return

    metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
    metric_labels = {
        "ExecTime_ms": "Tempo de Execução Médio (ms)",
        "MaxWorking_MB": "Memória Física Máx Média (MB)",
        "MaxPrivate_MB": "Memória Privada Máx Média (MB)",
        "AvgCPU_pct": "Uso de CPU Médio (%)",
        "AsmLines": "Linhas Assembly Médias"
    }

    # 1. Agrupar os dados pela Linguagem e pelo Teste para obter a média
    #    (Isso ignora a coluna 'Optimization', calculando a média geral para cada Teste/Linguagem)
    agg_tests = df.groupby(["Test", "Language"])[metrics].mean().reset_index()

    print("=== MÉDIAS AGREGADAS POR TESTE E LINGUAGEM ===")
    print(agg_tests)
    print()

    # 2. Gerar gráficos para cada métrica
    for metric in metrics:
        if metric not in agg_tests.columns:
            continue
            
        # --- Gráfico de Barras para Comparação entre Testes ---
        # Mostrar o crescimento/diferença de uma métrica em diferentes testes
        fig_bar = px.bar(
            agg_tests,
            x="Test",                 # Teste no eixo X
            y=metric,                 # Métrica no eixo Y (a média)
            color="Language",
            barmode="group",
            labels={metric: metric_labels[metric], "Test": "Tipo de Teste"},
            title=f"Comparação Média: {metric_labels[metric]} por Tipo de Teste (C vs. Haskell)",
            # Para 'AsmLines', o crescimento é o que interessa:
            log_y=(metric == "AsmLines") # Pode ser útil para 'AsmLines' se a variação for grande
        )
        fig_bar.update_layout(
            template="plotly_white",
            font=dict(size=14),
            yaxis_title=metric_labels[metric],
            xaxis_title="Tipo de Teste"
        )
        fig_bar.show()

        # --- Gráfico de Linhas (Útil se os 'Testes' tivessem uma ordem lógica) ---
        # Manter o gráfico de linhas caso os testes (e.g., min1, min2, min3) tenham ordem
        fig_line = px.line(
            agg_tests,
            x="Test",
            y=metric,
            color="Language",
            markers=True,
            labels={metric: metric_labels[metric], "Test": "Tipo de Teste"},
            title=f"Comparação Média: {metric_labels[metric]} por Tipo de Teste (C vs. Haskell) - Linha",
        )
        fig_line.update_traces(mode="lines+markers")
        fig_line.update_layout(
            template="plotly_white",
            font=dict(size=14),
            yaxis_title=metric_labels[metric],
            xaxis_title="Tipo de Teste"
        )
        fig_line.show()


# === EXECUÇÃO ===
# Chame a nova função, passando o seu DataFrame completo (df)
compare_tests_aggregated(df)

=== MÉDIAS AGREGADAS POR TESTE E LINGUAGEM ===
  Test Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  \
0  mpf        c   188.862333       2.132467       0.436817    0.753500   
1  mpf       hs   183.846667       2.218333       3.276350    1.125500   
2  mpf       py   835.003333       6.352667       1.311733    2.663333   

   AsmLines  
0    246.25  
1   1214.25  
2      0.00  



In [7]:
# === CRIA COLUNA DE TIPO DE TESTE ===
def parse_test_type(test_name):
    if "tc" in test_name.lower():
        return "Time Complexity"
    elif "min" in test_name.lower():
        return "Minimal Complexity"
    elif "ops_complex" in test_name.lower():
        return "Operation Complexity"
    else:
        return "Other"

# ATENÇÃO: se necessário, adicione no fluxo principal:
df["TestType"] = df["Test"].apply(parse_test_type)

def compare_type_clean(test_type):
    # 1. Definição das métricas e rótulos
    metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
    metric_labels = {
        "ExecTime_ms": "Tempo de Execução (ms)",
        "MaxWorking_MB": "Memória Física Máx (MB)",
        "MaxPrivate_MB": "Memória Privada Máx (MB)",
        "AvgCPU_pct": "Uso de CPU (%)",
        "AsmLines": "Linhas Assembly"
    }
    
    # 2. Filtra por tipo de teste
    subset = df[df["TestType"] == test_type]
    
    if subset.empty:
        print(f"⚠️ Nenhum dado para '{test_type}'")
        return

    # 3. AGREGAÇÃO: média por (Test, Language)
    agg_subset = subset.groupby(["Test", "Language"])[metrics].mean().reset_index()

    # Exibe um resumo indicando que é a MÉDIA (para deixar claro)
    print(f"=== AGREGADO (MÉDIA) — {test_type} ===")
    print(agg_subset.head())

    # 4. Geração dos Gráficos (com indicação explícita de "média")
    for metric in metrics:
        fig = px.bar(
            agg_subset,
            x="Test",
            y=metric,
            color="Language",
            barmode="group",
            text_auto="%d" if metric == "AsmLines" else ".2s", 
            title=f"✅ {test_type} — {metric_labels[metric]} (C vs Haskell) — média por teste",
            color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
        )
        
        fig.update_layout(
            template="plotly_white",
            font=dict(size=12),
            xaxis_title="Testes",
            # Marca explicitamente que o eixo Y representa a média
            yaxis_title=f"{metric_labels[metric]} (média)",
            xaxis_tickangle=-45,
            yaxis_type='linear' 
        )
        
        fig.show()


# === EXECUTA PARA TODOS OS TIPOS ===
types = df["TestType"].unique()
for t in types:
    compare_type_clean(t)


=== AGREGADO (MÉDIA) — Other ===
  Test Language  ExecTime_ms  MaxWorking_MB  MaxPrivate_MB  AvgCPU_pct  \
0  mpf        c   188.862333       2.132467       0.436817    0.753500   
1  mpf       hs   183.846667       2.218333       3.276350    1.125500   
2  mpf       py   835.003333       6.352667       1.311733    2.663333   

   AsmLines  
0    246.25  
1   1214.25  
2      0.00  


In [8]:
# Variáveis e DataFrames (assumindo que 'df', 'metrics', e 'metric_labels' já estão definidos)
metrics = ["ExecTime_ms", "MaxWorking_MB", "MaxPrivate_MB", "AvgCPU_pct", "AsmLines"]
metric_labels = {
    "ExecTime_ms": "Tempo de Execução (ms)",
    "MaxWorking_MB": "Memória Física Máx (MB)",
    "MaxPrivate_MB": "Memória Privada Máx (MB)",
    "AvgCPU_pct": "Uso de CPU (%)",
    "AsmLines": "Linhas Assembly"
}

# --- 1. Gráfico de Linha: Desempenho (ExecTime) vs. Nível de Otimização, por Linguagem ---

# Objetivo: Mostrar como cada linguagem responde às otimizações.
# Agrega o tempo médio de execução por Nível de Otimização e Linguagem, 
# abrangendo todos os testes.
agg_time_opt = df.groupby(['Optimization', 'Language'])[metrics].mean().reset_index()

fig1 = px.line(
    agg_time_opt,
    x="Optimization",
    y="ExecTime_ms",
    color="Language",
    line_group="Language",
    markers=True,
    title="1. Impacto da Otimização no Tempo de Execução (Média Geral)",
)
fig1.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['ExecTime_ms'],
    xaxis_title="Nível de Otimização",
    font=dict(size=12)
)
fig1.show()


# --- 2. Gráfico de Linha: Complexidade do Código (AsmLines) vs. Nível de Otimização ---

# Objetivo: Mostrar se a otimização reduz ou aumenta o número de instruções geradas.
# Agrega AsmLines por Nível de Otimização e Linguagem.
agg_asm_opt = df.groupby(['Optimization', 'Language'])['AsmLines'].mean().reset_index()

fig2 = px.line(
    agg_asm_opt,
    x="Optimization",
    y="AsmLines",
    color="Language",
    line_group="Language",
    markers=True,
    title="2. Tendência do Número de Linhas Assembly vs. Nível de Otimização",
)
fig2.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['AsmLines'],
    xaxis_title="Nível de Otimização",
    font=dict(size=12)
)
fig2.show()


# --- 3. Gráfico de Linha: Uso de Memória (MaxWorking_MB) ao longo dos Testes ---

# Objetivo: Mostrar o consumo de recursos (Memória) para cada teste, em média.
# Agrega o consumo médio de memória por Teste e Linguagem.
agg_memory_test = df.groupby(['Test', 'Language'])['MaxWorking_MB'].mean().reset_index()

fig3 = px.line(
    agg_memory_test,
    x="Test",
    y="MaxWorking_MB",
    color="Language",
    line_group="Language",
    markers=True,
    title="3. Consumo de Memória (Média) ao Longo da Sequência de Testes",
)
fig3.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['MaxWorking_MB'],
    xaxis_title="Teste (Min1 a Min5 / Tc1 a TcX)",
    xaxis_tickangle=-45,
    font=dict(size=12)
)
fig3.show()

In [9]:
# --- VISUALIZACAO 4: VIOLIN PLOT (Distribuição do Tempo de Execução por Teste) ---

# Objetivo: Mostrar a variabilidade e a densidade do tempo de execução por teste.

fig4 = px.violin(
    df,
    y="AvgCPU_pct",
    x="Test",
    color="Language",
    box=True,  # Inclui box plot dentro do violino
    points="all", # Mostra todos os pontos
    title="4. Distribuição do Uso de CPU (%) por Teste e Linguagem (Violin Plot)",
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)
fig4.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['AvgCPU_pct'],
    xaxis_title="Teste",
    font=dict(size=12)
)
fig4.show()


In [23]:
# --- VISUALIZACAO 4: VIOLIN PLOT (Distribuição do Tempo de Execução por Teste) ---

# Objetivo: Mostrar a variabilidade e a densidade do tempo de execução por teste.

fig4 = px.violin(
    df,
    y="ExecTime_ms",
    x="Test",
    color="Language",
    box=True,  # Inclui box plot dentro do violino
    points="all", # Mostra todos os pontos
    title="4. Distribuição do Tempo de Execução (ms) por Teste e Linguagem (Violin Plot)",
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)
fig4.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['ExecTime_ms'],
    xaxis_title="Teste",
    font=dict(size=12),
    height=1200
)
fig4.show()


In [11]:
# --- VISUALIZACAO 4: VIOLIN PLOT (Distribuição do Tempo de Execução por Teste) ---

# Objetivo: Mostrar a variabilidade e a densidade do tempo de execução por teste.

fig4 = px.violin(
    df,
    y="MaxPrivate_MB",
    x="Test",
    color="Language",
    box=True,  # Inclui box plot dentro do violino
    points="all", # Mostra todos os pontos
    title="4. Distribuição da Memória Privada Máxima (MB) por Teste e Linguagem (Violin Plot)",
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)
fig4.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['MaxPrivate_MB'],
    xaxis_title="Teste",
    font=dict(size=12)
)
fig4.show()


In [12]:
# --- VISUALIZACAO 4: VIOLIN PLOT (Distribuição do Tempo de Execução por Teste) ---

# Objetivo: Mostrar a variabilidade e a densidade do tempo de execução por teste.

fig4 = px.violin(
    df,
    y="AsmLines",
    x="Test",
    color="Language",
    box=True,  # Inclui box plot dentro do violino
    points="all", # Mostra todos os pontos
    title="4. Distribuição da Quantidade de Linhas de Assembly por Teste e Linguagem (Violin Plot)",
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)
fig4.update_layout(
    template="plotly_white",
    yaxis_title=metric_labels['AsmLines'],
    xaxis_title="Teste",
    font=dict(size=12)
)
fig4.show()


In [13]:
# --- VISUALIZACAO 5 (separada): cria um scatter + trendline para cada Test individualmente ---

# Usa a variável `tests` já definida (lista/array de nomes de testes)
for test_name in tests:
    subset = df[df["Test"] == test_name]
    if subset.empty:
        print(f"⚠️ Sem dados para {test_name}")
        continue

    fig = px.scatter(
        subset,
        x="MaxWorking_MB",
        y="ExecTime_ms",
        color="Language",
        trendline="ols",
        title=f"5 — {test_name}: Tempo de Execução vs Memória (por linguagem)",
        labels={"ExecTime_ms": metric_labels["ExecTime_ms"], "MaxWorking_MB": metric_labels["MaxWorking_MB"]},
        color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
    )
    fig.update_layout(
        template="plotly_white",
        font=dict(size=12),
        height=420,
        width=760
    )
    fig.show()


In [14]:
# --- Preparação: Agregação por Média para o Mapa de Calor ---
heatmap_agg = df.groupby(['Test', 'Language', 'Optimization'])[metrics].mean().reset_index()

# --- VISUALIZACAO 6: HEATMAP - Linhas Assembly vs Teste & Otimização (Geral) ---
asm_heatmap = heatmap_agg.pivot_table(
    index='Test',
    columns=['Language', 'Optimization'],
    values='AsmLines'
)

# Flatten multiindex columns para legendas mais legíveis
asm_heatmap.columns = [f"{lang} — O{opt}" for lang, opt in asm_heatmap.columns]

fig6 = px.imshow(
    asm_heatmap,
    text_auto=True,
    color_continuous_scale="Plasma",
    aspect="auto",
    title="6. Mapa de Calor: Linhas Assembly (Complexidade) por Teste, Linguagem e Otimização",
    labels={'x': 'Linguagem — Nível de Otimização', 'y': 'Teste', 'color': f"{metric_labels['AsmLines']} (média)"}
)

# Ajustes nas legendas/labels/tick para ficar mais claro
fig6.update_layout(
    template="plotly_white",
    xaxis={'side': 'top'},
    height=550,
    font=dict(size=11)
)
fig6.update_xaxes(tickangle=-45)
# Personaliza hover e o título do colorbar (legenda de cor)
fig6.data[0].hovertemplate = 'Linguagem/Opt: %{x}<br>Teste: %{y}<br>Linhas Assembly (média): %{z}<extra></extra>'
fig6.data[0].colorbar.title.text = f"{metric_labels['AsmLines']} (média)"

fig6.show()


In [15]:
fig = px.scatter(
    df,
    x="Test",
    y="MaxWorking_MB",
    color="Language",
    size="AsmLines",
    hover_data=["Optimization", "TestType", "ExecTime_ms"],
    title="Scatter: Memória Física Máx (MB) por Teste",
    category_orders={"Test": list(tests)},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800", "HS": "#00cc96"}
)

fig.update_traces(marker=dict(opacity=0.8, sizemin=6), selector=dict(mode="markers"))
fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    yaxis_title=metric_labels["MaxWorking_MB"],
    height=480
)
fig.show()


In [16]:
fig = px.scatter(
    df,
    x="Test",
    y="AvgCPU_pct",
    color="Language",
    size="AsmLines",
    hover_data=["Optimization", "TestType", "ExecTime_ms"],
    title="Scatter: Uso Médio de CPU (%) por Teste",
    category_orders={"Test": list(tests)},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800", "HS": "#00cc96"}
)

fig.update_traces(marker=dict(opacity=0.8, sizemin=6), selector=dict(mode="markers"))
fig.update_layout(
    template="plotly_white",
    xaxis_tickangle=-45,
    yaxis_title=metric_labels["AvgCPU_pct"],
    height=480
)
fig.show()


In [17]:

# --- VISUALIZACAO 8: BOX PLOT - Variação do Uso de CPU (%) por Nível de Otimização ---

# Objetivo: Mostrar a distribuição (variação) do uso de CPU e identificar outliers.

fig8 = px.box(
    df,
    x="Test",
    y="AvgCPU_pct",
    color="Language",
    points="outliers", # Mostra apenas outliers (pontos extremos)
    title="8. Distribuição do Uso de CPU (%) por Otimização e Linguagem (Box Plot)",
    labels={"AvgCPU_pct": metric_labels["AvgCPU_pct"], "AvgCPU_pct": "Gasto de CPU (%)"},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)

fig8.update_layout(
    template="plotly_white",
    font=dict(size=12)
)
fig8.show()

In [22]:

# --- VISUALIZACAO 8: BOX PLOT - Variação do Uso de CPU (%) por Nível de Otimização ---

# Objetivo: Mostrar a distribuição (variação) do uso de CPU e identificar outliers.

fig8 = px.box(
    df,
    x="Optimization",
    y="AvgCPU_pct",
    color="Language",
    points="all", # Mostra apenas outliers (pontos extremos)
    title="8. Distribuição do Uso de CPU (%) por Otimização e Linguagem (Box Plot)",
    labels={"AvgCPU_pct": metric_labels["AvgCPU_pct"], "Optimization": "Nível de Otimização"},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"}
)

fig8.update_layout(
    template="plotly_white",
    font=dict(size=12),
    height=780
)
fig8.show()

In [19]:
# --- VISUALIZAÇÃO 11: HISTOGRAMA - Distribuição do Tempo de Execução (ms) ---

# Objetivo: Mostrar a frequência de ocorrência dos tempos de execução.
# Usamos 'histfunc="count"' para contar as ocorrências em cada bin.

fig11 = px.histogram(
    df,
    x="ExecTime_ms",
    color="Language",
    marginal="box", # Adiciona um Box Plot na margem para ver a mediana e outliers
    opacity=0.7,
    title="11. Histograma: Distribuição de Frequência do Tempo de Execução",
    labels={"ExecTime_ms": metric_labels["ExecTime_ms"]},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"},
    # O 'barmode' "overlay" é crucial para sobrepor os histogramas
    barmode="overlay" 
)

fig11.update_layout(
    template="plotly_white",
    yaxis_title="Contagem de Ocorrências",
    font=dict(size=12)
)
fig11.show()

In [20]:
# --- VISUALIZAÇÃO 12: HISTOGRAMA - Distribuição de Linhas Assembly ---

# Objetivo: Mostrar a frequência de diferentes tamanhos de código gerado.

fig12 = px.histogram(
    df,
    x="AsmLines",
    color="Language",
    marginal="rug", # Adiciona um gráfico de "tapete" na margem para ver a dispersão exata
    opacity=0.7,
    title="12. Histograma: Distribuição da Quantidade de Linhas Assembly",
    labels={"AsmLines": metric_labels["AsmLines"]},
    color_discrete_map={"C": "#0077cc", "Haskell": "#ff8800"},
    barmode="overlay"
)

fig12.update_layout(
    template="plotly_white",
    yaxis_title="Contagem de Ocorrências",
    font=dict(size=12)
)
fig12.show()

In [21]:
# Agregação necessária para o Heatmap (garante um valor médio Z por X/Y)
heatmap_agg = df.groupby(['Test', 'Optimization', 'Language'])[metrics].mean().reset_index()


# --- VISUALIZAÇÃO 13: HEATMAP - Tempo de Execução vs Teste & Otimização ---

# Objetivo: Mostrar onde o tempo de execução é mais alto/baixo.
# A cor representa o Tempo de Execução (ExecTime_ms).
# O gráfico é facetado pela Linguagem para comparação direta.

fig13 = px.density_heatmap(
    heatmap_agg,
    x="Test",
    y="Optimization",
    z="ExecTime_ms",
    histfunc="sum", # Sumariza o valor de ExecTime_ms
    facet_col="Language", # Separa C e Haskell
    title="13. Heatmap: Tempo de Execução (Média) por Teste e Otimização",
    labels={'ExecTime_ms': metric_labels['ExecTime_ms'], 'Optimization': 'Nível de Otimização'},
    color_continuous_scale=px.colors.sequential.Inferno
)

fig13.update_layout(
    template="plotly_white",
    xaxis={'side': 'bottom'},
    height=400,
    font=dict(size=12)
)
fig13.show()

